In [1]:
from utils import make_nanoribbon, hamiltonian
from utils.structure import guess_hexagon_center, find_overlap, delete_overlaps
from utils.plots import plot_with_center
from typing import Tuple
import numpy as np
import matplotlib.pyplot as plt
from ase.visualize import view
import sisl
from sisl.physics import RecursiveSI
import pandas as pd
from numba import njit
from scipy.sparse import csc_array

%load_ext line_profiler


# Print matrix

In [2]:
def pretty_print_columns(A, decimals=2, zero_repr="0"):
    """
    Print a 2D NumPy array with per-column alignment.
    Handles complex numbers, negatives, and zeros gracefully.
    """
    A = np.asarray(A)
    if A.ndim != 2:
        raise ValueError("Input must be a 2D array")

    rows, cols = A.shape
    is_complex = np.iscomplexobj(A)

    # Build a string matrix (formatted values)
    str_matrix = np.empty(A.shape, dtype=object)
    for i in range(rows):
        for j in range(cols):
            val = A[i, j]
            if val == 0:
                s = zero_repr
            elif is_complex:
                s = f"{val.real:.{decimals}f}{val.imag:+.{decimals}f}j"
            else:
                s = f"{val:.{decimals}f}"
            str_matrix[i, j] = s

    # Compute per-column widths
    col_widths = [max(len(str_matrix[i, j]) for i in range(rows)) for j in range(cols)]

    # Print rows with proper per-column alignment
    for i in range(rows):
        row_str = " ".join(str_matrix[i, j].rjust(col_widths[j]) for j in range(cols))
        print(row_str)

# Make device

In [ ]:
def make_device_nanoribbon(electrode: sisl.Geometry, center_size : int):
    assert center_size > 0, "must have a center size of at least 1 `electrode`"
    return electrode.tile(2+center_size, 0)


WIDTH = 5
LENGTH = 2
electrode = make_nanoribbon(WIDTH, LENGTH)
ribbon = make_device_nanoribbon(electrode, 1)

In [37]:
def _device_atoms(device, electrode, reset=False):
    """Change the left electrode and right electrode atoms"""
    if not reset:
        N = len(electrode)
        device.atoms[:N] = sisl.Atoms("N") # set left electrode atoms to N (blue)
        device.atoms[-N:] = sisl.Atoms("O") # set right electrode atoms to O (red)
    
    if reset:
        device.atoms[:] = sisl.Atoms("C") # set all atoms to C

def _set_electrode_atoms(device, electrode):
    _device_atoms(device, electrode, reset=False)

def _return_device(device, electrode):
    _device_atoms(device, electrode, reset=True)
    

def find_electrode_indices(device, num_electrodes):
    symbols = np.array([atom.symbol for atom in device.atoms])
    
    # oxygen corresponds to right electrode (red)
    idx_O = np.where(symbols == "O")[0]
    NO = len(idx_O) // num_electrodes # number of O atoms per electrode
    
    # nitrogen correpsonds to left electrode (blue)
    idx_N = np.where(symbols == "N")[0]
    NN = len(idx_N) // num_electrodes # number of N atoms per electrode
    
    left_electrodes = []
    right_electrodes = []
    for i in range(num_electrodes):
        left = idx_N[i*NN:(i+1)*NN]
        right = idx_O[i*NO:(i+1)*NO]
        # print(f"left : {left}")
        # print(f"right : {right}")
        left_electrodes.append(left)
        right_electrodes.append(right)
    return np.array(left_electrodes), np.array(right_electrodes)
    
    
    
def structure(nanoribbon: sisl.Geometry, electrode: sisl.geometry, *, repeat=3):
    assert repeat in [1, 2, 3], "repeat can only be 1, 2, or 3"
    _, origin_of_rotation = guess_hexagon_center(nanoribbon)
    axis_of_rotation = [0,0,1]
    
    
    RIBBON = nanoribbon.copy()
    _set_electrode_atoms(RIBBON, electrode)
    FINAL = RIBBON.copy()
    for i in range(1, repeat):
        angle = 60
        angle = angle*(-1) if i%2 == 0 else angle
        rotated_nanoribbon = RIBBON.rotate(angle=angle, v=axis_of_rotation, origin=origin_of_rotation)
        FINAL += rotated_nanoribbon
    
    # remove overlapping atoms
    FINAL = delete_overlaps(FINAL)
    
    # find the indices for the electrodes using the different atom species
    left_indices, right_indices = find_electrode_indices(FINAL, repeat)
    
    # return to original atom species -- Carbon -- for all atoms
    _return_device(FINAL, electrode)
    
    return FINAL, left_indices, right_indices

device, left_idx, right_idx = structure(ribbon, electrode, repeat=3)
device.plot(axes="xy")

       Initial number of atoms: 180
 Atoms after removing overlaps: 144


In [ ]:
atoms_style = []
for i, (l, r) in enumerate(zip(left_idx, right_idx)):
    atoms_style += [{"atoms": l, "size": i*0.3 + 0.5, "color": "blue"}]
    atoms_style += [{"atoms": r, "size": i*0.3 + 0.5, "color": "red"}]

device.plot(axes="xy", atoms_style=atoms_style)

In [ ]:
view(device.to.ase())

# Compute left/right self-energies for different k points and energies

In [40]:
def _direction(**kwargs):
    "infer direction of extending kvector"
    Nk = kwargs.get("Nk", 1)
    axis = kwargs.get("axis", 1)
    if isinstance(axis, int):
        if not axis in range(3): # 0, 1, or 2
            raise ValueError("'axis' must be  0,  1,  or  2.")
        d = [1, 1, 1]
        d[axis] = Nk
        return d, Nk
    else:
        raise ValueError("axis must be  int.")
        
@njit
def hermconj(matrix):
    "Hermitian conjugate of 2d matrix"
    assert matrix.ndim == 2, "matrix must be 2D"
    return matrix.T.conj()

@njit
def calc_gamma(se):
    "Compute left/right broadening matrix from left/right self-energy"
    return 1j*(se - hermconj(se))


@njit
def greens(left, right, E, H):
    "compute freens function from Energy, device hamiltonian and left/right self-energies."
    assert left.shape == right.shape, "left and right self-energies not identical."
    N = len(left)
    invG = E - H
    invG[ :N,  :N] -= left
    invG[-N:, -N:] -= right
    return np.linalg.inv(invG)

@njit
def spectral(Greens, Gamma):
    "compute left/right spectral function from greens function and left/right broadening matrix"
    return Greens @ Gamma @ hermconj(Greens)

@njit
def _T(AR, GammaL):
    return np.trace(AR @ GammaL)


def lr_energies(device, electrode, **kwargs):
    energies = kwargs.get("energies", 0)
    eta = kwargs.get("eta", 1e-5)
    if isinstance(energies, (float, int)): # convert single value energies to list 
        energies = [energies]
        
    
    NE = len(energies)
    N_device = len(device)
    N_electrode = len(electrode)
    k_direction, Nk = _direction(**kwargs)
    kpts = sisl.MonkhorstPack(device, k_direction).k
    kwargs.setdefault("kpts", kpts)
    kwargs.setdefault("energies", energies)
    SE = RecursiveSI(electrode, "+A", eta=eta)
    # greens_kE = np.zeros(shape=(Nk, NE, N_device, N_device), dtype=complex)
    left_energies = np.empty(shape=(Nk, NE, N_electrode, N_electrode), dtype=complex)
    right_energies = left_energies.copy()
    for iter_k, kvec in enumerate(kpts):
        # Hk = device.Hk(k=kvec, format="array").astype(complex)
        # Sk = device.Sk(k=kvec, format="array").astype(complex)
        
        for iter_E, E in enumerate(energies):
            En = E + 1j*eta
            SE_L, SE_R = SE.self_energy_lr(E=En)
            # greens_kE[iter_k,iter_E, ...] = greens(left=SE_L, right=SE_R, E=En*Sk, H=Hk)
            left_energies[iter_k, iter_E, ...] = SE_L
            right_energies[iter_k, iter_E, ...] = SE_R
    return left_energies, right_energies, kwargs


In [41]:
H_D = hamiltonian(ribbon)
H_0 = hamiltonian(electrode)
L_energy, R_energy, _ = lr_energies(H_D, H_0)

In [ ]:
def device_hamiltonian(device, ribbon, electrode, **kwargs):
    ribbon_HAM = hamiltonian(ribbon)
    electrode_HAM = hamiltonian(electrode)
    device_HAM = hamiltonian(device)
    device_HAM.set_nsc((1,1,1)) # no PBC for structure
    
    left_energies, right_energies, kwargs = lr_energies(ribbon_HAM, electrode_HAM, **kwargs)
    assert left_energies.shape == right_energies.shape, "dimensions of left and right energies does not mathc..."
    device = device.copy()
    electode = electode.copy()
    
    num_k, num_E = left_energies.shape[:2]
    hams = np.empty_like(left_energies)
    
    for iter_k, kvec in enumerate(kwargs.get("kpts")):
        Hk = device_HAM.Hk(kvec=kvec, format="array").astype(complex)
        for iter_E, E in enumerate(kwargs.get("energies")):
            hams[iter_k, iter_E, ...] = Hk + left_energies[iter_k, iter_E] + right_energies[iter_k, iter_E]
    
    return hams
            
            
    

In [347]:
def LDOS(device, electrode, energies, **kwargs):
    T, A_ek, kpts = lr_energies(device, electrode, energies, **kwargs)
    print(f"{A_ek.shape = }")
    assert A_ek.shape[1] == len(energies), "number of energies and corresponding dimension of A does not match (check implementation)"
    
    def rho(kidx, Eidx):
        """Find LDOS from diagonal of spectral function/matrix"""
        return np.diag(A_ek[kidx, Eidx]).real / (2*np.pi)
    
    LDOS = np.zeros(shape=(*A_ek.shape[:2], A_ek.shape[-1]), dtype=float) # number of (k, E, A.shape) 
    for iter_E, E in zip(range(A_ek.shape[1]), energies):
        for iter_k in range(A_ek.shape[0]):
            LDOS[iter_k, iter_E, :] = rho(iter_k, iter_E)
    
    return T, LDOS, kpts

In [348]:
Nk = 1
NE = 50
energies = np.linspace(-1, 1, num=NE)
T, ldos, kpts = LDOS(H_D, H_0, energies); print(ldos.shape)

TypeError: lr_energies() takes 2 positional arguments but 3 were given

In [ ]:
import ipywidgets
from ipywidgets import interact
from fractions import Fraction

def plotLDOS(ldos, kidx, site):
    if isinstance(kidx, ipywidgets.Dropdown):
        kidx = kidx.value
    fig, ax = plt.subplots(1,1)
    
    ymin, ymax = np.min(ldos), np.max(ldos)
    
    ax.plot(energies, ldos[kidx, :, site])
    ax.set_ylim(-ymax*0.1, ymax*1.01)
    ax.set_xlabel("E")
    ax.set_ylabel("LDOS")
    
    # ax.legend()
options = [(f"{[str(Fraction(val).limit_denominator(len(kpts)*2)) for val in kvec]})", i) for i, kvec in enumerate(kpts)]
ks = ipywidgets.Dropdown(options=options, description="k")
static = ipywidgets.fixed(ldos)
sites = ipywidgets.IntSlider(min=0, max=ldos.shape[-1]-1, value=0, description="Site")

interact(plotLDOS, ldos=static, kidx=ks, site=sites)
    

interactive(children=(Dropdown(description='k', options=(("['0', '0', '0'])", 0),), value=0), IntSlider(value=…

<function __main__.plotLDOS(ldos, kidx, site)>

### Profiling

In [ ]:
def inverse1(G):
    return np.linalg.inv(G)
def inverse2(G):
    I = np.eye(G.shape[0])
    return np.linalg.solve(G, I)
def device_hamiltonian(G):
    inverse1(G)
    inverse2(G)    
    # return np.linalg.solve(G, I)
%lprun -f func func(G)

Timer unit: 1e-09 s

Total time: 0.0629423 s
File: /tmp/ipykernel_282758/1137498103.py
Function: func at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def func(G):
     7         1   11338174.0 1.13e+07     18.0      inverse1(G)
     8         1   51604130.0 5.16e+07     82.0      inverse2(G)    
     9                                               # return np.linalg.solve(G, I)

In [ ]:
%lprun -f transport -s -u 1e-3 LDOS(PBC_ham, energies)

A_ek.shape = (1, 11, 300, 300)


Timer unit: 0.001 s

Total time: 29.6675 s
File: /tmp/ipykernel_282758/1335168255.py
Function: transport at line 41

Line #      Hits         Time  Per Hit   % Time  Line Contents
    41                                           def transport(H, energies, **kwargs):
    42         1          0.0      0.0      0.0      eta = kwargs.get("eta", 1e-5)
    43         1          0.0      0.0      0.0      k_direction, Nk = _direction(**kwargs)
    44                                           
    45                                           
    46         1          0.8      0.8      0.0      kpts = sisl.MonkhorstPack(H, k_direction).k
    47         1          0.0      0.0      0.0      NE = len(energies)
    48         1          0.0      0.0      0.0      T_k_sum = np.zeros(shape=(Nk, NE), dtype=float)
    49         1          0.3      0.3      0.0      A_ek = np.empty(shape=(Nk, NE, *H.Hk(format="array").shape), dtype=complex)
    50                                           
    51   